# Notebook 2: Tokenization and Embeddings

## What You'll Learn
- How real LLMs convert text to tokens (Byte Pair Encoding)
- Build a BPE tokenizer from scratch
- What word embeddings are and why they matter
- How Word2Vec works — train your own word vectors
- Positional encodings: how models know word order

---

## Why This Matters

In Notebook 1, we used **character-level** tokenization. Real LLMs use **subword** tokenization — a sweet spot between characters and words. Understanding tokenization is critical because:

1. It determines the model's **vocabulary** and what it can represent
2. It affects **training efficiency** (fewer tokens = faster training)
3. Tokenization bugs cause real failures in production LLMs
4. The embedding layer is the **first thing** the model sees

In [ ]:
# Install dependencies
# !pip install torch numpy matplotlib tiktoken

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import re

torch.manual_seed(42)
np.random.seed(42)

## 1. The Tokenization Problem

### Why Not Characters?
- Sequences become very long ("artificial" = 10 tokens)
- Model must learn spelling from scratch
- Wastes context window on individual characters

### Why Not Words?
- Vocabulary explodes (100k+ unique words)
- Can't handle new/rare words ("COVID-19" in 2019)
- Can't handle typos, other languages, code

### The Solution: Subword Tokenization
Split text into **subword units** that balance vocabulary size and sequence length.

Common words stay whole: `"the"` → `["the"]`  
Rare words get split: `"tokenization"` → `["token", "ization"]`  
Very rare words split more: `"pneumonoultramicroscopicsilicovolcanoconiosis"` → many pieces

## 2. Byte Pair Encoding (BPE) — From Scratch

BPE is the tokenization algorithm used by GPT-2, GPT-3, GPT-4, and many other LLMs.

### Algorithm
1. Start with individual characters (or bytes) as the initial vocabulary
2. Count all adjacent pairs in the training data
3. Merge the most frequent pair into a new token
4. Repeat steps 2-3 until desired vocabulary size is reached

Let's implement this step by step.

In [ ]:
# Training corpus for our BPE tokenizer
corpus = [
    "the cat sat on the mat",
    "the dog sat on the log",
    "the cat and the dog played",
    "cats and dogs are playing together",
    "the cat is sitting on the mat again",
    "dogs love to play with cats",
    "the mat is where the cat likes to sit",
    "a dog and a cat sat together on the mat",
    "playing and sitting are the things cats do",
    "the dog loves the cat and the cat loves the dog",
]

print(f'Corpus: {len(corpus)} sentences')
print(f'Total characters: {sum(len(s) for s in corpus)}')

In [ ]:
class BPETokenizer:
    """Byte Pair Encoding tokenizer built from scratch."""
    
    def __init__(self):
        self.merges = {}  # (pair) -> new_token
        self.vocab = {}   # token_id -> token_string
    
    def _get_word_freqs(self, corpus):
        """Split corpus into words and count frequencies."""
        word_freqs = Counter()
        for sentence in corpus:
            words = sentence.split()
            for word in words:
                # Add end-of-word marker
                word_freqs[' '.join(list(word)) + ' </w>'] += 1
        return word_freqs
    
    def _get_pair_counts(self, word_freqs):
        """Count adjacent pairs across all words."""
        pairs = Counter()
        for word, freq in word_freqs.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i+1])] += freq
        return pairs
    
    def _merge_pair(self, word_freqs, pair):
        """Merge a pair of symbols in all words."""
        new_word_freqs = {}
        bigram = ' '.join(pair)
        replacement = ''.join(pair)
        
        for word, freq in word_freqs.items():
            new_word = word.replace(bigram, replacement)
            new_word_freqs[new_word] = freq
        
        return new_word_freqs
    
    def train(self, corpus, num_merges=20, verbose=True):
        """Train BPE on a corpus."""
        word_freqs = self._get_word_freqs(corpus)
        
        if verbose:
            print("Initial vocabulary (characters):")
            all_chars = set()
            for word in word_freqs:
                for sym in word.split():
                    all_chars.add(sym)
            print(f"  {sorted(all_chars)}")
            print(f"  Vocab size: {len(all_chars)}")
            print()
        
        merge_history = []
        
        for i in range(num_merges):
            pair_counts = self._get_pair_counts(word_freqs)
            if not pair_counts:
                break
            
            # Find most frequent pair
            best_pair = max(pair_counts, key=pair_counts.get)
            best_count = pair_counts[best_pair]
            
            # Merge it
            word_freqs = self._merge_pair(word_freqs, best_pair)
            
            merged_token = ''.join(best_pair)
            self.merges[best_pair] = merged_token
            merge_history.append((best_pair, best_count))
            
            if verbose:
                print(f"Merge {i+1:2d}: '{best_pair[0]}' + '{best_pair[1]}' -> '{merged_token}' (count: {best_count})")
        
        # Build final vocabulary
        all_tokens = set()
        for word in word_freqs:
            for sym in word.split():
                all_tokens.add(sym)
        
        self.vocab = {i: token for i, token in enumerate(sorted(all_tokens))}
        self.token_to_id = {token: i for i, token in self.vocab.items()}
        
        if verbose:
            print(f"\nFinal vocab size: {len(self.vocab)}")
            print(f"Final vocabulary: {sorted(self.vocab.values())}")
        
        return word_freqs, merge_history
    
    def encode(self, text):
        """Encode text into token IDs."""
        words = text.split()
        tokens = []
        
        for word in words:
            # Start with characters
            symbols = list(word) + ['</w>']
            
            # Apply merges in order
            for pair, merged in self.merges.items():
                i = 0
                new_symbols = []
                while i < len(symbols):
                    if i < len(symbols) - 1 and symbols[i] == pair[0] and symbols[i+1] == pair[1]:
                        new_symbols.append(merged)
                        i += 2
                    else:
                        new_symbols.append(symbols[i])
                        i += 1
                symbols = new_symbols
            
            for sym in symbols:
                if sym in self.token_to_id:
                    tokens.append(self.token_to_id[sym])
        
        return tokens
    
    def decode(self, ids):
        """Decode token IDs back to text."""
        tokens = [self.vocab[i] for i in ids]
        text = ''.join(tokens)
        text = text.replace('</w>', ' ')
        return text.strip()

# Train our BPE tokenizer
tokenizer = BPETokenizer()
final_words, history = tokenizer.train(corpus, num_merges=25)

In [ ]:
# Visualize the merge process
pairs = [f"{p[0]}+{p[1]}" for p, c in history]
counts = [c for p, c in history]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(range(len(pairs)), counts, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(pairs)))
ax.set_xticklabels(pairs, rotation=45, ha='right', fontsize=9)
ax.set_xlabel('Merge Operation')
ax.set_ylabel('Frequency')
ax.set_title('BPE Merge Operations (ordered by when they were applied)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Test encoding and decoding
test_sentences = [
    "the cat sat on the mat",
    "a dog is playing",
    "cats love sitting",
]

for sentence in test_sentences:
    encoded = tokenizer.encode(sentence)
    decoded = tokenizer.decode(encoded)
    token_strs = [tokenizer.vocab[i] for i in encoded]
    
    print(f'Input:    "{sentence}"')
    print(f'Tokens:   {token_strs}')
    print(f'IDs:      {encoded}')
    print(f'Decoded:  "{decoded}"')
    print(f'Chars: {len(sentence)} -> Tokens: {len(encoded)} (compression: {len(sentence)/len(encoded):.1f}x)')
    print()

## 3. Real-World Tokenizers

Let's explore the tokenizer used by GPT models. OpenAI's `tiktoken` library gives us access to the actual tokenizers.

In [ ]:
import tiktoken

# GPT-4 / GPT-3.5-turbo tokenizer
enc = tiktoken.get_encoding("cl100k_base")

test_texts = [
    "Hello, world!",
    "The quick brown fox jumps over the lazy dog.",
    "Machine learning is fascinating.",
    "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
    "🤖 Artificial Intelligence",
    "pneumonoultramicroscopicsilicovolcanoconiosis",
]

for text in test_texts:
    tokens = enc.encode(text)
    token_strs = [enc.decode([t]) for t in tokens]
    print(f'Text: "{text}"')
    print(f'  Tokens ({len(tokens)}): {token_strs}')
    print()

In [ ]:
# Interesting tokenization edge cases
# These reveal important properties of how LLMs "see" text

edge_cases = [
    # Spaces matter!
    (" hello", "hello"),  # leading space is a different token
    # Numbers
    ("123", "12345", "123456789"),
    # Whitespace
    ("a  b", "a   b", "a    b"),
]

print("=== Tokenization Edge Cases ===")
print()

for group in edge_cases:
    for text in group:
        tokens = enc.encode(text)
        token_strs = [repr(enc.decode([t])) for t in tokens]
        print(f'  {repr(text):25s} -> {len(tokens)} tokens: {token_strs}')
    print()

print("Key insight: LLMs don't see text the way you do.")
print("They see a sequence of these subword tokens.")
print("This is why LLMs struggle with character-level tasks like counting letters!")

## 4. Word Embeddings: From IDs to Meaning

Token IDs are just arbitrary numbers. The model needs to convert them into **dense vector representations** that capture meaning.

### What is an Embedding?

An embedding maps each token to a point in a high-dimensional space:

$$\text{embedding}: \text{token\_id} \rightarrow \mathbb{R}^d$$

Where $d$ is the embedding dimension (typically 768 to 12,288 in real LLMs).

### Why Embeddings Work

During training, the model learns to place similar tokens **close together** in embedding space:
- "king" and "queen" end up nearby
- "cat" and "dog" end up nearby  
- "cat" and "seven" end up far apart

This is an **emergent property** — nobody tells the model which words are similar.

In [ ]:
# Let's build and train Word2Vec (Skip-gram) from scratch
# This is the classic algorithm that showed embeddings capture meaning

# Training data: a larger corpus
training_text = """
the king sat on the throne in the castle
the queen sat beside the king on her throne
the prince and princess lived in the castle
a man and a woman walked through the town
the boy and girl played in the garden
the king ruled the kingdom with the queen
the man worked in the town every day
the woman walked to the castle to see the queen
the prince will be the next king of the kingdom
the princess will be the next queen
cats and dogs are common pets
the cat sat on the mat in the house
the dog ran in the garden and played
a kitten is a young cat
a puppy is a young dog
cats like to sit and sleep all day
dogs like to run and play all day
the kitten played with the puppy in the garden
paris is the capital of france
london is the capital of england
berlin is the capital of germany
tokyo is the capital of japan
france and germany are in europe
japan is in asia
england is in europe
people in paris speak french
people in london speak english
people in berlin speak german
people in tokyo speak japanese
""".strip().lower()

# Tokenize (word-level for Word2Vec)
words = training_text.split()
vocab = sorted(set(words))
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for i, w in enumerate(vocab)}
vocab_size = len(vocab)

print(f'Vocabulary: {vocab_size} words')
print(f'Total tokens: {len(words)}')
print(f'Sample words: {vocab[:20]}')

In [ ]:
# Build Skip-gram training pairs
# For each word, predict surrounding words within a window

window_size = 3
skip_gram_pairs = []

for i, center_word in enumerate(words):
    center_idx = word_to_idx[center_word]
    
    # Look at words within the window
    for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
        if i != j:  # skip the center word itself
            context_idx = word_to_idx[words[j]]
            skip_gram_pairs.append((center_idx, context_idx))

print(f'Training pairs: {len(skip_gram_pairs)}')
print(f'\nSample pairs (center -> context):')
for center, context in skip_gram_pairs[:10]:
    print(f'  "{idx_to_word[center]}" -> "{idx_to_word[context]}"')

In [ ]:
# Word2Vec Skip-gram Model

class Word2Vec(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        # Two embedding matrices:
        # center_embeddings: for when a word is the center word
        # context_embeddings: for when a word is a context word
        self.center_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)
        
        # Initialize with small random values
        nn.init.xavier_uniform_(self.center_embeddings.weight)
        nn.init.xavier_uniform_(self.context_embeddings.weight)
    
    def forward(self, center_ids, context_ids):
        # Get embeddings
        center_emb = self.center_embeddings(center_ids)    # (batch, embed_dim)
        context_emb = self.context_embeddings(context_ids)  # (batch, embed_dim)
        
        # Dot product: how much does center predict context?
        scores = torch.sum(center_emb * context_emb, dim=-1)  # (batch,)
        return scores
    
    def get_embeddings(self):
        """Return the center embeddings as the final word vectors."""
        return self.center_embeddings.weight.detach()

# Create model
embedding_dim = 32
w2v_model = Word2Vec(vocab_size, embedding_dim)
print(f'Model parameters: {sum(p.numel() for p in w2v_model.parameters()):,}')

In [ ]:
# Train Word2Vec with negative sampling

optimizer = torch.optim.Adam(w2v_model.parameters(), lr=0.01)
num_negatives = 5  # For each positive pair, sample 5 negative pairs

# Convert to tensors
centers = torch.tensor([p[0] for p in skip_gram_pairs])
contexts = torch.tensor([p[1] for p in skip_gram_pairs])

losses = []
batch_size = 256

for epoch in range(100):
    # Shuffle
    perm = torch.randperm(len(centers))
    centers_shuffled = centers[perm]
    contexts_shuffled = contexts[perm]
    
    epoch_loss = 0
    num_batches = 0
    
    for i in range(0, len(centers), batch_size):
        batch_centers = centers_shuffled[i:i+batch_size]
        batch_contexts = contexts_shuffled[i:i+batch_size]
        
        # Positive pairs: these words DO appear together
        pos_scores = w2v_model(batch_centers, batch_contexts)
        pos_loss = -F.logsigmoid(pos_scores).mean()
        
        # Negative pairs: random words that probably DON'T appear together
        neg_contexts = torch.randint(0, vocab_size, (len(batch_centers), num_negatives))
        neg_scores = w2v_model(
            batch_centers.unsqueeze(1).expand_as(neg_contexts).reshape(-1),
            neg_contexts.reshape(-1)
        ).view(len(batch_centers), num_negatives)
        neg_loss = -F.logsigmoid(-neg_scores).mean()
        
        loss = pos_loss + neg_loss
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    losses.append(avg_loss)
    
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:3d} | Loss: {avg_loss:.4f}')

plt.figure(figsize=(10, 4))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Word2Vec Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Explore the learned embeddings!

def find_similar(word, top_k=5):
    """Find the most similar words by cosine similarity."""
    embeddings = w2v_model.get_embeddings()
    word_idx = word_to_idx[word]
    word_emb = embeddings[word_idx]
    
    # Cosine similarity with all words
    similarities = F.cosine_similarity(
        word_emb.unsqueeze(0), embeddings, dim=1
    )
    
    # Get top-k (excluding the word itself)
    values, indices = similarities.topk(top_k + 1)
    
    results = []
    for val, idx in zip(values, indices):
        if idx.item() != word_idx:
            results.append((idx_to_word[idx.item()], val.item()))
    
    return results[:top_k]

# Test semantic similarity
test_words = ['king', 'cat', 'paris', 'dog', 'france']

for word in test_words:
    similar = find_similar(word)
    print(f'Words most similar to "{word}":')
    for sim_word, score in similar:
        print(f'  {sim_word:15s} {score:.3f}')
    print()

In [ ]:
# Visualize embeddings with t-SNE / PCA
from sklearn.decomposition import PCA

embeddings = w2v_model.get_embeddings().numpy()
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

# Color-code by semantic category
categories = {
    'royalty': ['king', 'queen', 'prince', 'princess', 'throne', 'kingdom', 'castle', 'ruled'],
    'animals': ['cat', 'dog', 'cats', 'dogs', 'kitten', 'puppy', 'pets'],
    'places': ['paris', 'london', 'berlin', 'tokyo', 'france', 'england', 'germany', 'japan', 'europe', 'asia'],
    'people': ['man', 'woman', 'boy', 'girl', 'people'],
}

plt.figure(figsize=(14, 10))

# Plot uncategorized words in gray
categorized = set(w for ws in categories.values() for w in ws)
for i, word in enumerate(vocab):
    if word not in categorized:
        plt.scatter(embeddings_2d[i, 0], embeddings_2d[i, 1], c='lightgray', s=30, alpha=0.5)
        plt.annotate(word, (embeddings_2d[i, 0], embeddings_2d[i, 1]), fontsize=7, alpha=0.4)

# Plot categorized words with colors
colors = {'royalty': 'gold', 'animals': 'green', 'places': 'blue', 'people': 'red'}
for category, words_list in categories.items():
    for word in words_list:
        if word in word_to_idx:
            idx = word_to_idx[word]
            plt.scatter(embeddings_2d[idx, 0], embeddings_2d[idx, 1], 
                       c=colors[category], s=100, alpha=0.8, label=category)
            plt.annotate(word, (embeddings_2d[idx, 0], embeddings_2d[idx, 1]), 
                        fontsize=10, fontweight='bold')

# Deduplicate legend
handles, labels = plt.gca().get_legend_handles_labels()
by_label = dict(zip(labels, handles))
plt.legend(by_label.values(), by_label.keys(), fontsize=12)

plt.title('Word Embeddings Projected to 2D (PCA)', fontsize=14)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Notice how semantically similar words cluster together!")
print("The model learned these relationships purely from word co-occurrence patterns.")

In [ ]:
# Word analogies: king - man + woman = queen?

def analogy(a, b, c, top_k=3):
    """Find d such that a:b :: c:d
    i.e., a - b + c ≈ d"""
    embeddings = w2v_model.get_embeddings()
    
    vec = embeddings[word_to_idx[a]] - embeddings[word_to_idx[b]] + embeddings[word_to_idx[c]]
    
    similarities = F.cosine_similarity(vec.unsqueeze(0), embeddings, dim=1)
    
    # Exclude input words
    exclude = {word_to_idx[a], word_to_idx[b], word_to_idx[c]}
    
    values, indices = similarities.topk(top_k + len(exclude))
    results = []
    for val, idx in zip(values, indices):
        if idx.item() not in exclude:
            results.append((idx_to_word[idx.item()], val.item()))
    
    return results[:top_k]

# Test analogies (results may vary with small dataset)
analogies = [
    ('king', 'man', 'woman'),      # king - man + woman = ?
    ('paris', 'france', 'japan'),   # paris - france + japan = ?
    ('cat', 'kitten', 'puppy'),     # cat - kitten + puppy = ?
]

for a, b, c in analogies:
    results = analogy(a, b, c)
    print(f'{a} - {b} + {c} = ?')
    for word, score in results:
        print(f'  {word:15s} ({score:.3f})')
    print()

## 5. Positional Encodings

Embeddings tell the model **what** each token is, but not **where** it is in the sequence. Transformers process all tokens in parallel, so they need explicit position information.

### Sinusoidal Positional Encoding (from "Attention Is All You Need")

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

Where:
- $pos$ = position in the sequence
- $i$ = dimension index
- $d$ = embedding dimension

### Why Sinusoidal?
- Each position gets a unique encoding
- The model can learn to attend to relative positions (because $PE_{pos+k}$ can be expressed as a linear function of $PE_{pos}$)
- Generalizes to sequence lengths not seen during training

In [ ]:
# Implement and visualize positional encodings

def sinusoidal_positional_encoding(max_len, d_model):
    """Create sinusoidal positional encodings."""
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    
    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * -(np.log(10000.0) / d_model)
    )
    
    pe[:, 0::2] = torch.sin(position * div_term)  # even dimensions
    pe[:, 1::2] = torch.cos(position * div_term)  # odd dimensions
    
    return pe

# Generate positional encodings
max_len = 100
d_model = 64
pe = sinusoidal_positional_encoding(max_len, d_model)

print(f'Positional encoding shape: {pe.shape}')
print(f'  (max_sequence_length x embedding_dimension)')

In [ ]:
# Visualize the positional encodings
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Heatmap of all positional encodings
im = axes[0].imshow(pe.numpy().T, aspect='auto', cmap='RdBu_r', interpolation='nearest')
axes[0].set_xlabel('Position in Sequence')
axes[0].set_ylabel('Encoding Dimension')
axes[0].set_title('Sinusoidal Positional Encodings')
plt.colorbar(im, ax=axes[0])

# Plot a few specific dimensions
dims_to_plot = [0, 1, 4, 5, 10, 11, 20, 21]
for dim in dims_to_plot:
    axes[1].plot(pe[:, dim].numpy(), label=f'dim {dim}', alpha=0.7)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Encoding Value')
axes[1].set_title('Individual Encoding Dimensions')
axes[1].legend(fontsize=8, ncol=4)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Each position gets a unique 'fingerprint' across dimensions.")
print("Low dimensions oscillate slowly (capture coarse position).")
print("High dimensions oscillate quickly (capture fine position).")

In [ ]:
# Show that nearby positions have similar encodings
# (measured by cosine similarity)

position_similarities = F.cosine_similarity(
    pe.unsqueeze(0),  # (1, max_len, d_model)
    pe.unsqueeze(1),  # (max_len, 1, d_model)
    dim=2
)

plt.figure(figsize=(10, 8))
plt.imshow(position_similarities[:50, :50].numpy(), cmap='viridis')
plt.colorbar(label='Cosine Similarity')
plt.xlabel('Position')
plt.ylabel('Position')
plt.title('Cosine Similarity Between Positional Encodings')
plt.tight_layout()
plt.show()

print("Nearby positions have high similarity (bright diagonal).")
print("This lets the model learn distance-based attention patterns.")

## 6. Putting It All Together: The Input Pipeline

In a real LLM, the input to the Transformer is:

$$\text{Input} = \text{TokenEmbedding}(x) + \text{PositionalEncoding}(\text{position})$$

Let's build this complete input pipeline.

In [ ]:
class LLMInputPipeline(nn.Module):
    """Complete input pipeline for a Transformer language model."""
    
    def __init__(self, vocab_size, d_model, max_seq_len, use_learned_pos=False):
        super().__init__()
        self.d_model = d_model
        
        # Token embedding
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        
        # Position embedding (learned vs sinusoidal)
        if use_learned_pos:
            # GPT-style: learned positional embeddings
            self.position_embedding = nn.Embedding(max_seq_len, d_model)
        else:
            # Original Transformer: sinusoidal (not learned)
            pe = sinusoidal_positional_encoding(max_seq_len, d_model)
            self.register_buffer('pe', pe)
        
        self.use_learned_pos = use_learned_pos
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, token_ids):
        """
        token_ids: (batch_size, seq_len) - integer token IDs
        returns: (batch_size, seq_len, d_model) - embeddings ready for Transformer
        """
        batch_size, seq_len = token_ids.shape
        
        # Token embeddings (scaled by sqrt(d_model) as in the paper)
        tok_emb = self.token_embedding(token_ids) * np.sqrt(self.d_model)
        
        # Position embeddings
        if self.use_learned_pos:
            positions = torch.arange(seq_len, device=token_ids.device)
            pos_emb = self.position_embedding(positions)
        else:
            pos_emb = self.pe[:seq_len]
        
        # Add them together
        x = tok_emb + pos_emb
        x = self.dropout(x)
        
        return x

# Demo
pipeline = LLMInputPipeline(
    vocab_size=50000,   # Typical LLM vocab
    d_model=128,        # Small for demo
    max_seq_len=512,
    use_learned_pos=True  # GPT-style
)

# Simulate some token IDs
fake_tokens = torch.randint(0, 50000, (2, 20))  # batch of 2, length 20
output = pipeline(fake_tokens)

print(f'Input token IDs shape:  {fake_tokens.shape}')
print(f'Output embeddings shape: {output.shape}')
print(f'\nThis output goes into the Transformer layers (Notebook 3)!')
print(f'\nPipeline parameters: {sum(p.numel() for p in pipeline.parameters()):,}')

## 7. Exercises

### Exercise 1: Implement Your Own BPE
Take a larger corpus (download a book from Project Gutenberg) and train BPE with different vocabulary sizes (100, 500, 1000, 5000). Plot the average tokens per sentence vs vocabulary size. What's the optimal tradeoff?

In [ ]:
# EXERCISE 1: BPE with different vocab sizes
# Your code here

### Exercise 2: Embedding Arithmetic
Train Word2Vec on a larger corpus and test more word analogies. Can you find:
- Country-capital relationships
- Gender analogies
- Verb tense relationships

In [ ]:
# EXERCISE 2: More word analogies
# Your code here

### Exercise 3: Rotary Positional Embeddings (RoPE)
Modern LLMs (LLaMA, Mistral) use RoPE instead of sinusoidal or learned positions. Research and implement RoPE. Key idea: instead of adding position info, it **rotates** the embedding vectors.

$$\text{RoPE}(x, m) = \begin{pmatrix} x_1 \cos m\theta - x_2 \sin m\theta \\ x_1 \sin m\theta + x_2 \cos m\theta \end{pmatrix}$$

In [ ]:
# EXERCISE 3: Implement RoPE
# Your code here

## 8. Key Takeaways

1. **BPE tokenization** is a compression algorithm that builds a vocabulary by repeatedly merging frequent character pairs
2. **Tokenization affects model behavior** — LLMs literally "see" text differently than humans do
3. **Embeddings map tokens to dense vectors** where similar tokens are close in vector space
4. **Word2Vec showed that embeddings capture semantic relationships** (king - man + woman ≈ queen)
5. **Positional encodings** tell the model where each token is in the sequence
6. The **input to a Transformer** is: token embedding + positional encoding

### Next Up
**Notebook 3: The Transformer Architecture** — We'll build the core of every modern LLM: self-attention, multi-head attention, and the full Transformer block from scratch.